# 🐍 Part 2: Database Enrichment & Data Wrangling via Python
## Refactoring and Enriching the `Addresses` Table (`01_update_addresses.ipynb`)

## 🎯 Context & Objectives

The goal of this stage is to perform a strategic data enrichment on the `addresses` table within the `insight_places` database using Python (`01_update_addresses.ipynb`). To make the database more realistic for advanced spatial analytical queries and BI dashboards, I will refactor the geographic layer by inserting new attributes, ensuring spatial consistency, and adding neighborhood granularity for drill-down reporting.

This notebook represents the first module of a sequential ETL execution, maintaining complete data integrity and referential constraints across the database ecosystem.



## 🛠️ Environment Setup & Dependencies

To execute this ETL pipeline securely and efficiently, the following Python libraries are required:

* **`sqlalchemy` & `pymysql`**: Database abstraction layer and MySQL driver for executing DDL/DML operations.
* **`python-dotenv`**: Security protocol to parse database credentials from the existing `.env` environment file without exposing sensitive keys.
* **`pandas`**: Data manipulation and spatial validation check.
* **`random`**: Native module used to apply geographic conditional logic and weighted distributions.



## 📌 Refining the `Addresses` Table

The first step focuses on transforming the `addresses` table into a robust spatial dimension:

1. **Real City Mapping:** Standardizing cities by replacing synthetic placeholder names with verified Brazilian municipalities mapped to their respective state (`state`).
2. **New Attribute (`location_type`):** Adding a classification column (`Beach`, `Mountain`, `Countryside`, `City`, `Near Airport`).
3. **Geographic Business Rules:**
   * `Beach` category is strictly restricted to coastal states (e.g., RJ, SP, SC, BA, CE).
   * Inland states (e.g., MG, GO, MT) are restricted to `Mountain`, `Countryside`, `City`, or `Near Airport`.
4. **Neighborhood & Street Enrichment:** Updating `neighborhood` and `street` with realistic tourist districts (e.g., Copacabana, Ipanema, Jardins) to enable district-level pricing and performance analysis in Power BI.
5. **Capital Expansion:** Inserting missing records for the Federal District (`DF` - Brasília).

_____________________________________________________________________________________________________________________________________________________________________________________________
### 1) Installing and importing the libraries

In this step, I will load the libraries that will be used throughout the project.



In [3]:
!pip install sqlalchemy
!pip install pymysql

import pandas as pd
import random
from sqlalchemy import create_engine, text
import pymysql

#### 1.1) Installing the data protection library
I will install the `python-dotenv` library to ensure the security of the MySQL password.

In [4]:
!pip install python-dotenv

#### 1.2) Environment Setup and Security Validation

In [5]:
# Installing and Importing libraries
import os
from dotenv import load_dotenv

# Loading variables from the .env file
load_dotenv()

# Assigning credentials to variables using os.getenv
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '3306')
database = os.getenv('DB_NAME', 'insight_places')

# Security Check: Validating if environment variables are present
if all([user, password, host, database]):
    print("🔒 Environment variables loaded successfully.")
    print(" System ready for database connection.")
else:
    print("❌ Error: One or more credentials are missing from the .env file.")

  
    

🔒 Environment variables loaded successfully.
 System ready for database connection.


#### 1.3) Database Engine Initialization
Using the validated credentials, I will use SQLAlchemy to create the engine. This object will be the communication bridge between Python and MySQL, allowing for efficient data transfer.

In [6]:
# Creating the database connection engine using validated credentials
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")
print("🚀 Engine connection is ready!")

🚀 Engine connection is ready!


## 2) Execution: Refining and Enriching the `Addresses` Table

In [7]:
# Real Brazilian cities mapped by State
STATE_CITIES = {
    "RJ": ["Rio de Janeiro", "Niterói", "Búzios", "Petrópolis", "Angra dos Reis", "Cabo Frio"],
    "SP": ["São Paulo", "Campinas", "Santos", "Ubatuba", "Campos do Jordão", "Barueri"],
    "MG": ["Belo Horizonte", "Ouro Preto", "Monte Verde", "Tiradentes", "Poços de Caldas"],
    "SC": ["Florianópolis", "Balneário Camboriú", "Bombinhas", "Joinville", "Blumenau"],
    "BA": ["Salvador", "Porto Seguro", "Praia do Forte", "Ilhéus", "Trancoso"],
    "CE": ["Fortaleza", "Jericoacoara", "Canoa Quebrada", "Aquiraz"],
    "PE": ["Recife", "Porto de Galinhas", "Olinda", "Fernando de Noronha"],
    "RS": ["Porto Alegre", "Gramado", "Canela", "Bento Gonçalves"],
    "PR": ["Curitiba", "Foz do Iguaçu", "Matinhos", "Cascavel"],
    "GO": ["Goiânia", "Pirenópolis", "Caldas Novas", "Rio Quente"],
    "ES": ["Vitória", "Vila Velha", "Guarapari", "Domingos Martins"],
    "DF": ["Brasília"],
    "AM": ["Manaus", "Parintins"],
    "PA": ["Belém", "Santarém", "Alter do Chão"],
    "RN": ["Natal", "Pipa"],
    "PB": ["João Pessoa", "Campina Grande"],
    "AL": ["Maceió", "Maragogi"],
    "SE": ["Aracaju"],
    "MA": ["São Luís", "Barreirinhas"],
    "MT": ["Cuiabá", "Chapada dos Guimarães"],
    "MS": ["Campo Grande", "Bonito"],
    "TO": ["Palmas", "Jalapão"],
    "RO": ["Porto Velho"],
    "AC": ["Rio Branco"],
    "AP": ["Macapá"],
    "RR": ["Boa Vista"],
    "PI": ["Teresina", "Barra Grande"]
}

# Coastal states allowed to have 'Beach'
COASTAL_STATES = {
    "RJ", "SP", "SC", "BA", "CE", "PE", "RN", "PB", "AL", "ES", "PR", "SE", "MA", "AP", "PA", "PI"
}

def update_addresses():
    with engine.begin() as conn:
        print("1. Adding 'location_type' column to 'addresses' table...")
        # Safe DDL using try/except for column existence
        try:
            conn.execute(text("""
                ALTER TABLE addresses 
                ADD COLUMN location_type VARCHAR(50) AFTER state;
            """))
            print("   Column 'location_type' successfully created.")
        except Exception as e:
            # Code 1060 = Duplicate column name in MySQL
            if "1060" in str(e):
                print("   Column 'location_type' already exists. Proceeding...")
            else:
                raise e

        print("2. Fetching existing address records...")
        result = conn.execute(text("SELECT address_id, state FROM addresses"))
        addresses = result.fetchall()

        print("3. Updating existing addresses with real cities and valid location types...")
        for address_id, state in addresses:
            cities = STATE_CITIES.get(state, ["Capital"])
            chosen_city = random.choice(cities)

            # Determine allowed location types based on state geography
            if state in COASTAL_STATES:
                if chosen_city in ["Búzios", "Angra dos Reis", "Cabo Frio", "Santos", "Ubatuba", "Florianópolis", "Balneário Camboriú", "Bombinhas", "Porto Seguro", "Praia do Forte", "Trancoso", "Jericoacoara", "Porto de Galinhas", "Guarapari", "Pipa", "Maragogi", "Alter do Chão"]:
                    location_type = "Beach"
                else:
                    location_type = random.choice(["Beach", "City", "Near Airport"])
            elif state in ["MG", "RJ", "SP", "RS", "GO", "MT", "MS"] and chosen_city in ["Petrópolis", "Campos do Jordão", "Monte Verde", "Tiradentes", "Poços de Caldas", "Gramado", "Canela", "Pirenópolis", "Chapada dos Guimarães", "Bonito"]:
                location_type = random.choice(["Mountain", "Countryside"])
            else:
                location_type = random.choice(["City", "Near Airport", "Countryside"])

            conn.execute(
                text("""
                    UPDATE addresses 
                    SET city = :city, location_type = :location_type 
                    WHERE address_id = :address_id
                """),
                {"city": chosen_city, "location_type": location_type, "address_id": address_id}
            )

        print("4. Inserting new addresses for Brasília (DF)...")
        df_check = conn.execute(text("SELECT COUNT(*) FROM addresses WHERE state = 'DF'")).scalar()
        
        if df_check == 0:
            df_addresses = [
                {"street": "Setor Hoteleiro Norte (SHN)", "number": "100", "city": "Brasília", "state": "DF", "zip_code": "70705-000", "location_type": "City"},
                {"street": "Setor de Hotéis e Turismo Norte (SHTN)", "number": "250", "city": "Brasília", "state": "DF", "zip_code": "70800-200", "location_type": "Near Airport"},
                {"street": "Lago Norte, MI Trecho 3", "number": "45", "city": "Brasília", "state": "DF", "zip_code": "71500-000", "location_type": "Countryside"}
            ]
            for addr in df_addresses:
                conn.execute(
                    text("""
                        INSERT INTO addresses (street, number, city, state, zip_code, location_type)
                        VALUES (:street, :number, :city, :state, :zip_code, :location_type)
                    """),
                    addr
                )

        print("✅ Addresses table updated successfully!")

# Run the updated function
update_addresses()

1. Adding 'location_type' column to 'addresses' table...
   Column 'location_type' successfully created.
2. Fetching existing address records...
3. Updating existing addresses with real cities and valid location types...
4. Inserting new addresses for Brasília (DF)...
✅ Addresses table updated successfully!


### 2.1) Neighborhood and Street Enrichment
To enhance spatial granularity for future Power BI analyses (drill-down by district and average daily rate comparisons), I update the `neighborhood` and `street` columns with realistic urban data mapped to key tourist hubs.

In [9]:
# Mapping key neighborhoods and streets for major Brazilian tourist destinations
CITY_DISTRICTS = {
    "Rio de Janeiro": [
        {"neighborhood": "Copacabana", "street": "Avenida Atlântica"},
        {"neighborhood": "Ipanema", "street": "Rua Visconde de Pirajá"},
        {"neighborhood": "Leblon", "street": "Avenida Delfim Moreira"},
        {"neighborhood": "Botafogo", "street": "Rua Voluntários da Pátria"},
        {"neighborhood": "Santa Teresa", "street": "Rua Paschoal Carlos Magno"},
        {"neighborhood": "Barra da Tijuca", "street": "Avenida Lúcio Costa"},
        {"neighborhood": "Lapa", "street": "Rua do Lavradio"}
    ],
    "São Paulo": [
        {"neighborhood": "Jardins", "street": "Alameda Lorena"},
        {"neighborhood": "Vila Madalena", "street": "Rua Harmonia"},
        {"neighborhood": "Pinheiros", "street": "Rua dos Pinheiros"},
        {"neighborhood": "Itaim Bibi", "street": "Rua João Cachoeira"},
        {"neighborhood": "Moema", "street": "Alameda dos Maracatins"},
        {"neighborhood": "Bela Vista", "street": "Avenida Paulista"}
    ],
    "Salvador": [
        {"neighborhood": "Barra", "street": "Avenida Oceânica"},
        {"neighborhood": "Pelourinho", "street": "Rua Gregório de Mattos"},
        {"neighborhood": "Rio Vermelho", "street": "Rua da Paciência"},
        {"neighborhood": "Ondina", "street": "Avenida Mário Leal Ferreira"}
    ],
    "Florianópolis": [
        {"neighborhood": "Jurerê Internacional", "street": "Avenida dos Búzios"},
        {"neighborhood": "Lagoa da Conceição", "street": "Rua das Rendeiras"},
        {"neighborhood": "Centro", "street": "Avenida Beira Mar Norte"},
        {"neighborhood": "Canasvieiras", "street": "Rua das Gaivotas"}
    ],
    "Gramado": [
        {"neighborhood": "Centro", "street": "Avenida Borges de Medeiros"},
        {"neighborhood": "Planalto", "street": "Rua João Petry"},
        {"neighborhood": "Bavária", "street": "Rua das Flamboyants"}
    ],
    "Brasília": [
        {"neighborhood": "Asa Sul", "street": "SQS 208"},
        {"neighborhood": "Asa Norte", "street": "SQN 312"},
        {"neighborhood": "Setor Hoteleiro Norte", "street": "SHN Quadra 2"},
        {"neighborhood": "Lago Sul", "street": "SHIS QL 10"}
    ]
}

# Generic fallback districts for other cities
GENERIC_DISTRICTS = [
    {"neighborhood": "Centro", "street": "Rua Principal"},
    {"neighborhood": "Jardim América", "street": "Avenida Brasil"},
    {"neighborhood": "Bela Vista", "street": "Rua das Flores"},
    {"neighborhood": "Zona Sul", "street": "Avenida Beira Mar"}
]

def update_neighborhoods_and_streets():
    with engine.begin() as conn:
        print("1. Fetching addresses to enrich neighborhoods and streets...")
        result = conn.execute(text("SELECT address_id, city FROM addresses"))
        records = result.fetchall()

        print("2. Updating records with realistic spatial data...")
        for address_id, city in records:
            if city in CITY_DISTRICTS:
                district_info = random.choice(CITY_DISTRICTS[city])
            else:
                district_info = random.choice(GENERIC_DISTRICTS)

            conn.execute(
                text("""
                    UPDATE addresses 
                    SET neighborhood = :neighborhood, street = :street 
                    WHERE address_id = :address_id
                """),
                {
                    "neighborhood": district_info["neighborhood"],
                    "street": district_info["street"],
                    "address_id": address_id
                }
            )

        print("✅ Neighborhoods and streets updated successfully!")

# Run enrichment
update_neighborhoods_and_streets()

1. Fetching addresses to enrich neighborhoods and streets...
2. Updating records with realistic spatial data...
✅ Neighborhoods and streets updated successfully!


## 3) Data Quality & Validation Check
To ensure the refactoring was executed correctly, I perform two validation queries:
1. Verifying the total distribution of `location_type`.
2. Validating spatial consistency (ensuring no `Beach` categories exist in inland states).

In [10]:
# Query 1: Distribution of location types
query_distribution = """
SELECT location_type, COUNT(*) as total_records
FROM addresses
GROUP BY location_type
ORDER BY total_records DESC;
"""

df_distribution = pd.read_sql(query_distribution, engine)
print("--- Location Type Distribution ---")
print(df_distribution)

print("\n" + "="*40 + "\n")

# Query 2: Checking geographical rule integrity strictly for landlocked / inland states
query_inland_check = """
SELECT state, city, location_type, COUNT(*) as count
FROM addresses
WHERE state IN ('MG', 'GO', 'DF', 'MT', 'MS', 'TO', 'AM', 'RO', 'AC', 'RR')
  AND location_type = 'Beach'
GROUP BY state, city, location_type;
"""

df_inland_check = pd.read_sql(query_inland_check, engine)

if df_inland_check.empty:
    print("✅ Geographic integrity verified: Zero inland addresses assigned as 'Beach'!")
else:
    print("⚠️ Warning: Geographic rule violation detected:")
    print(df_inland_check)

--- Location Type Distribution ---
  location_type  total_records
0         Beach           3018
1          City           2581
2  Near Airport           2451
3   Countryside           1532
4      Mountain            402


✅ Geographic integrity verified: Zero inland addresses assigned as 'Beach'!
